### Environment Setup

#### Virtual Environment

Create and activate a virtual environment:

```bash
python -m venv venv
source venv/bin/activate  # On Windows: venv\Scripts\activate
```

#### Install Dependencies

Install required packages:

```bash
pip install -r requirements.txt
```

### Import Required Libraries

Import all necessary libraries for data processing, model training, and evaluation.

In [219]:
import torch
import numpy as np
from datasets import Dataset, ClassLabel
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

### Load Dataset

Load the Tamil abuse detection dataset from CSV file.

In [220]:
from datasets import load_dataset
dataset = load_dataset(
"csv",
data_files="trainV2.csv"
)


---
#### Data Preprocessing

Normalize the data by:
- Removing whitespace at the end of labels
- Converting labels to consistent case (handling variations like "abusive" vs "Abusive")
---

In [221]:
label2id = {
    "Non-Abusive": 0,
    "Abusive": 1
}
def encode_labels(example):
    label = example["Class"].strip()
    if label.lower() == "abusive":
        example["Class"] = label2id["Abusive"]
    elif label.lower() == "non-abusive":
        example["Class"] = label2id["Non-Abusive"]
    else:
        # Fallback: try direct lookup
        example["Class"] = label2id[label]
    return example
dataset = dataset.map(encode_labels)

#### Rename Columns

Rename columns to match the expected format for model training:
- `Text` → `text`
- `Class` → `label`

In [222]:
dataset = dataset.rename_column("Text", "text")
dataset = dataset.rename_column("Class", "label")

#### Rename Columns
- Encoding string labels to integer IDs (0: Non-Abusive, 1: Abusive) 

In [223]:
def convert_label(example):
    example["label"] = int(example["label"])
    return example

dataset = dataset.map(convert_label)

In [224]:
dataset["train"][:5]

{'text': ['நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சிட்டேன் திவ்யா. நீ தெளிவாதான் இருக்க. உங்க வீடியோ பாக்குரேன்ல நான்தான் பைத்தியமா இருந்துருக்கேன்',
  'உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங்கமா தாண்டி இருக்கும்',
  'கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்து வையுங்கள்',
  'ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ ஒனக்கு இந்த சைஸ் போதுமா?',
  'ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷோ நடத்தி டெவலப் பண்றீயா'],
 'label': ['0', '1', '0', '1', '0']}

### Cast to ClassLabel for Stratification

convert the label from char array to int array

In [225]:
from datasets import ClassLabel
dataset = dataset.cast_column("label", ClassLabel(num_classes=2, names=["Non-Abusive", "Abusive"]))


In [226]:
dataset["train"][:5]

{'text': ['நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சிட்டேன் திவ்யா. நீ தெளிவாதான் இருக்க. உங்க வீடியோ பாக்குரேன்ல நான்தான் பைத்தியமா இருந்துருக்கேன்',
  'உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங்கமா தாண்டி இருக்கும்',
  'கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்து வையுங்கள்',
  'ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ ஒனக்கு இந்த சைஸ் போதுமா?',
  'ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷோ நடத்தி டெவலப் பண்றீயா'],
 'label': [0, 1, 0, 1, 0]}

### Dataset Statistics

Analyze the label distribution in the dataset to understand class balance.

In [227]:
from collections import Counter

# Count labels in the full dataset
label_counts = Counter(dataset["train"]["label"])
print("Dataset Label Distribution:")
print(f"Non-Abusive (0): {label_counts.get(0, 0)}")
print(f"Abusive (1): {label_counts.get(1, 0)}")
print(f"Total: {sum(label_counts.values())}")

Dataset Label Distribution:
Non-Abusive (0): 1883
Abusive (1): 1769
Total: 3652


### Train/Test Split

Split the dataset into training (80%) and test (20%) sets with stratification to maintain class distribution in both splits.

In [228]:
dataset_split = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="label"
)
train_dataset = dataset_split["train"]
test_dataset = dataset_split["test"]

### Training Set Statistics

Analyze the label distribution in the training set.

In [229]:
# Count labels in training set
train_label_counts = Counter(train_dataset["label"])
print("Training Set Label Distribution:")
print(f"Non-Abusive (0): {train_label_counts.get(0, 0)}")
print(f"Abusive (1): {train_label_counts.get(1, 0)}")
print(f"Total: {sum(train_label_counts.values())}")

Training Set Label Distribution:
Non-Abusive (0): 1506
Abusive (1): 1415
Total: 2921


### Test Set Statistics

Analyze the label distribution in the test set.

In [230]:
# Count labels in test set
test_label_counts = Counter(test_dataset["label"])
print("Test Set Label Distribution:")
print(f"Non-Abusive (0): {test_label_counts.get(0, 0)}")
print(f"Abusive (1): {test_label_counts.get(1, 0)}")
print(f"Total: {sum(test_label_counts.values())}")

Test Set Label Distribution:
Non-Abusive (0): 377
Abusive (1): 354
Total: 731
